# Faz 3a — YOLO Veri Hazırlık
### manifest.csv · splits · det_data PNG export

Bu notebook **Faz3_YOLO_Colab_Kaggle.ipynb** çalıştırılmadan önce bir kez çalıştırılır.

| Adım | Açıklama | Süre |
|------|----------|------|
| manifest.csv | DICOM metadata → CSV | ~5 dk |
| splits | 5-fold **StratifiedGroupKFold** + stratifiye hold-out | <1 dk |
| det_data (1 fold) | PNG kesit export | ~30–90 dk |
| det_data (5 fold) | Tüm foldlar | ~3–6 saat |

**Split stratejisi:**
- Negatif vakalar (238) `has_*=0` etiketiyle eğitim havuzuna dahil edilir
- Hold-out nadir sınıfa (divertikülit) göre stratifiye edilir
- `StratifiedGroupKFold`: hem vaka sızıntısı yok hem sınıf dengesi var
- **Hold-out skorlama**: 5 fold modelinin ensemble average'ı (olasılıklar ortalaması)

> **Not:** `EXPORT_ALL_FOLDS = False` yaparsanız yalnızca `FOLD` değişkenindeki fold export edilir.

---
## 0. Ortam Tespiti

In [1]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f'Ortam : {env_name}')

Ortam : Local


---
## 1. Ortam Kurulumu (Colab için)

In [2]:
if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print('Kaggle kimlik bilgileri Colab Secrets\'tan yüklendi')
    except Exception:
        from google.colab import files
        import json as _j, shutil as _sh
        print('kaggle.json dosyasını seçin:')
        _up = files.upload()
        _kp = Path.home() / '.kaggle' / 'kaggle.json'
        _kp.parent.mkdir(parents=True, exist_ok=True)
        for fname in _up:
            _sh.move(fname, str(_kp))
        os.chmod(str(_kp), 0o600)
        _kd = _j.loads(_kp.read_text())
        os.environ['KAGGLE_USERNAME'] = _kd['username']
        os.environ['KAGGLE_KEY']      = _kd['key']
        print('kaggle.json yüklendi')

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive bağlandı')
else:
    print('Kaggle/Local ortamı — Colab kurulumu atlandı')

Kaggle/Local ortamı — Colab kurulumu atlandı


---
## 2. Kütüphane Kurulumu

In [3]:
print('Kütüphaneler kuruluyor...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'pandas', 'numpy', 'tqdm', 'pillow', 'openpyxl',
     'pydicom', 'scikit-learn', 'python-dotenv'],
    check=True
)
import importlib; importlib.invalidate_caches()

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
print('Kütüphaneler hazır ✓')

Kütüphaneler kuruluyor...
Kütüphaneler hazır ✓


---
## 3. Konfigürasyon

In [4]:
import os, json
from pathlib import Path
from typing import List

# ─── Kullanıcı Ayarları ───────────────────────────────────────────────────
KAGGLE_DATASET_SLUG = 'abdomen-acikveri'
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen-acikveri'
GITHUB_URL          = 'https://github.com/ramazan2020/abdomen1.git'

FOLD             = 0      # Varsayılan fold (EXPORT_ALL_FOLDS=False ise bu fold export edilir)
N_FOLDS          = 5      # Toplam fold sayısı
EXPORT_ALL_FOLDS = False   # True: tüm 5 fold export edilir
# ─────────────────────────────────────────────────────────────────────────

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

if IS_KAGGLE:
    DATA_DIR   = Path('/kaggle/input') / KAGGLE_DATASET_SLUG
    WORK_DIR   = Path('/kaggle/working')

elif IS_COLAB:
    DATA_DIR   = Path('/content/kaggle_data')
    WORK_DIR   = Path('/content')

elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    _proje   = Path(os.environ.get('TR_ABDOMEN_PROJE',  r'D:/makale-pdf/Proje'))
    DATA_DIR = Path(os.environ.get('TR_ABDOMEN_BASE',   str(_proje / 'abdomen')))
    WORK_DIR = Path(os.environ.get('ABDOMEN_OUT_DIR',   str(_proje / 'outputs')))

SPLIT_DIR    = Path(os.environ.get('ABDOMEN_SPLIT_DIR',   str(WORK_DIR / 'splits')))
DET_DATA_DIR = Path(os.environ.get('ABDOMEN_DET_DATA_DIR', str(WORK_DIR / 'det_data')))

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
DET_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Ortam        : {env_name}')
print(f'DATA_DIR     : {DATA_DIR}  (var={DATA_DIR.exists()})')
print(f'WORK_DIR     : {WORK_DIR}  (var={WORK_DIR.exists()})')
print(f'SPLIT_DIR    : {SPLIT_DIR}  (var={SPLIT_DIR.exists()})')
print(f'DET_DATA_DIR : {DET_DATA_DIR}  (var={DET_DATA_DIR.exists()})')

if not DATA_DIR.exists():
    raise FileNotFoundError(f'DATA_DIR bulunamadı: {DATA_DIR}')

Ortam        : Local
DATA_DIR     : /Users/ramazanpolat/Desktop/datasets/abdomenDataSet  (var=True)
WORK_DIR     : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs  (var=True)
SPLIT_DIR    : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/splits  (var=True)
DET_DATA_DIR : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/det_data  (var=True)


---
## 4. GitHub Repo / Local src

In [5]:
if IS_LOCAL:
    REPO_DIR = Path('.').resolve()
    print(f'Local: src/ kullanılıyor → {REPO_DIR / "src"}')
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        print(f'Klonlanıyor: {GITHUB_URL}')
        subprocess.run(
            ['git', 'clone', '--depth=1', GITHUB_URL, str(REPO_DIR)],
            check=True
        )
    else:
        print('Repo mevcut, güncelleniyor...')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       capture_output=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# env var'larını src'nin görebileceği şekilde ayarla
os.environ.setdefault('ABDOMEN_SPLIT_DIR',    str(SPLIT_DIR))
os.environ.setdefault('ABDOMEN_DET_DATA_DIR', str(DET_DATA_DIR))
os.environ.setdefault('ABDOMEN_BILGI_XLSX',   str(DATA_DIR / 'Bilgi.xlsx'))

print(f'Repo : {REPO_DIR}')
print(f'sys.path[0] = {sys.path[0]}')

Local: src/ kullanılıyor → /Users/ramazanpolat/Desktop/datasets/abdomen/src
Repo : /Users/ramazanpolat/Desktop/datasets/abdomen
sys.path[0] = /Users/ramazanpolat/Desktop/datasets/abdomen


---
## 5. manifest.csv

In [6]:
_manifest_csv = SPLIT_DIR / 'manifest.csv'

if _manifest_csv.exists():
    manifest_df = pd.read_csv(_manifest_csv)
    print(f'manifest.csv mevcut ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')
    print(f'  {len(manifest_df):,} satır · {manifest_df["case"].nunique()} vaka')
else:
    print('manifest.csv bulunamadı — oluşturuluyor...')
    _bilgi = Path(os.environ.get('ABDOMEN_BILGI_XLSX', str(DATA_DIR / 'Bilgi.xlsx')))
    if not _bilgi.exists():
        raise FileNotFoundError(
            f'Bilgi.xlsx bulunamadı: {_bilgi}\n'
            'Faz1_2_VeriHazirlik notebook\'unu önce çalıştırın.'
        )
    from src.preprocessing import build_manifest
    build_manifest(_manifest_csv)
    manifest_df = pd.read_csv(_manifest_csv)
    print(f'manifest.csv oluşturuldu ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')
    print(f'  {len(manifest_df):,} satır · {manifest_df["case"].nunique()} vaka')

print(f'\nKolonlar: {manifest_df.columns.tolist()}')

manifest.csv mevcut ✓  (5083 KB)
  39,268 satır · 1092 vaka

Kolonlar: ['case', 'image_id', 'source', 'dicom_path', 'super_labels', 'bboxes', 'anatomical_boundary', 'n_bbox_anns', 'n_boundary_anns']


---
## 6. Splits (5-Fold StratifiedGroupKFold + Negatifler)

`REGEN_SPLITS = True` → eski splits.csv silinir, yeni mantıkla (negatifler dahil, stratifiye) yeniden üretilir.

In [7]:
from src.splits import (make_splits, _make_strata, _load_negative_cases,
                        _vaka_super_matrix, load_merged_annotations)

REGEN_SPLITS = True   # splits mantığı değişti — ilk çalıştırmada True bırakın

_splits_csv  = SPLIT_DIR / 'splits.csv'
_holdout_csv = SPLIT_DIR / 'holdout.csv'

if REGEN_SPLITS:
    for _f in list(SPLIT_DIR.glob('fold*_*.csv')) + [_splits_csv, _holdout_csv]:
        if _f.exists():
            _f.unlink()
    print('Eski split dosyaları silindi — yeniden oluşturuluyor...')
    make_splits(out_dir=SPLIT_DIR)
    print()

splits_df  = pd.read_csv(_splits_csv)
holdout_df = pd.read_csv(_holdout_csv)

# Vaka sayıları
_all_manifest = set(pd.read_csv(SPLIT_DIR / 'manifest.csv')['case'].unique())
_annotated    = set(load_merged_annotations()['Case Number'].unique())
n_pos   = len(_annotated)
n_neg   = len(_all_manifest) - n_pos
n_total = len(splits_df) + len(holdout_df)

print(f'Toplam vaka     : {n_total}  ({n_pos} pozitif + {n_neg} negatif)')
print(f'Hold-out (test) : {len(holdout_df)} vaka  ({len(holdout_df)/n_total*100:.1f}%)')
print(f'Train+Val havuzu: {len(splits_df)} vaka  ({len(splits_df)/n_total*100:.1f}%)')
print()

# Fold dağılımı
for f in range(N_FOLDS):
    n_tr = len(pd.read_csv(SPLIT_DIR / f'fold{f}_train.csv'))
    n_vl = len(pd.read_csv(SPLIT_DIR / f'fold{f}_val.csv'))
    print(f'  fold{f}: {n_tr:>5} train ({n_tr/n_total*100:.1f}%)  '
          f'{n_vl:>4} val ({n_vl/n_total*100:.1f}%) ✓')
print()

# Hold-out stratum dağılımı (nadir sınıf temsili doğrulaması)
_STRATA_NAMES = {
    0: 'negatif',
    6: 'acute_diverticulitis (en nadir)',
    5: 'aortic_aneurysm_dissection',
    4: 'acute_appendicitis',
    3: 'acute_pancreatitis',
    2: 'acute_cholecystitis',
    1: 'kidney_ureter_stone',
}
_case_mat = _vaka_super_matrix(load_merged_annotations())
_neg_mat  = _load_negative_cases(_case_mat, SPLIT_DIR)
if not _neg_mat.empty:
    _case_mat = pd.concat([_case_mat, _neg_mat], ignore_index=True)
_strata_map = dict(zip(_case_mat['Case Number'], _make_strata(_case_mat)))

ho_strata = [_strata_map.get(c, 0) for c in holdout_df['Case Number']]
print('Hold-out stratum dağılımı (stratifiye ✓):')
for s, cnt in sorted(pd.Series(ho_strata).value_counts().items()):
    print(f'  stratum {s}  {_STRATA_NAMES.get(s, str(s)):<35} {cnt:>3} vaka')

Eski split dosyaları silindi — yeniden oluşturuluyor...
Splits: 854 pozitif + 238 negatif = 1092 toplam vaka

Toplam vaka     : 1092  (1092 pozitif + 0 negatif)
Hold-out (test) : 164 vaka  (15.0%)
Train+Val havuzu: 928 vaka  (85.0%)

  fold0:   742 train (67.9%)   186 val (17.0%) ✓
  fold1:   742 train (67.9%)   186 val (17.0%) ✓
  fold2:   743 train (68.0%)   185 val (16.9%) ✓
  fold3:   743 train (68.0%)   185 val (16.9%) ✓
  fold4:   742 train (67.9%)   186 val (17.0%) ✓

Hold-out stratum dağılımı (stratifiye ✓):
  stratum 0  negatif                              36 vaka
  stratum 1  kidney_ureter_stone                  28 vaka
  stratum 2  acute_cholecystitis                  25 vaka
  stratum 3  acute_pancreatitis                   28 vaka
  stratum 4  acute_appendicitis                   13 vaka
  stratum 5  aortic_aneurysm_dissection           31 vaka
  stratum 6  acute_diverticulitis (en nadir)       3 vaka


---
## 7. YOLO det_data — PNG Export

`EXPORT_ALL_FOLDS = True` → 5 fold birden export edilir.  
`EXPORT_ALL_FOLDS = False` → yalnızca `FOLD` değişkenindeki fold export edilir.

In [8]:
import time
from src.detection import export_yolo_dataset

folds_to_export = list(range(N_FOLDS)) if EXPORT_ALL_FOLDS else [FOLD]
print(f'Export edilecek foldlar : {folds_to_export}')
print(f'Hedef dizin             : {DET_DATA_DIR}')
print(f'Negatif vakalar dahil   : True (boş label → false positive bastırma)')
print()

for fold in folds_to_export:
    fold_dir  = DET_DATA_DIR / f'fold{fold}'
    train_img = fold_dir / 'images' / 'train'
    val_img   = fold_dir / 'images' / 'val'

    n_train = len(list(train_img.glob('*.png'))) if train_img.exists() else 0
    n_val   = len(list(val_img.glob('*.png')))   if val_img.exists()   else 0

    if n_train > 0:
        print(f'Fold {fold}: mevcut ✓  ({n_train:,} train · {n_val:,} val PNG)')
    else:
        print(f'Fold {fold}: oluşturuluyor...')
        t0 = time.time()
        export_yolo_dataset(
            fold=fold,
            include_val_negatives=False,
            bbox_only=True,
            include_train_negatives=True,   # negatif vakalar boş label ile train'e girer
        )
        elapsed = time.time() - t0
        n_train = len(list(train_img.glob('*.png'))) if train_img.exists() else 0
        n_val   = len(list(val_img.glob('*.png')))   if val_img.exists()   else 0
        print(f'Fold {fold}: oluşturuldu ✓  ({n_train:,} train · {n_val:,} val PNG — {elapsed/60:.1f} dk)')

Export edilecek foldlar : [0]
Hedef dizin             : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/det_data
Negatif vakalar dahil   : True (boş label → false positive bastırma)

Fold 0: mevcut ✓  (20,964 train · 5,470 val PNG)


### 8. Sınıf Dağılımı — Train / Val / Holdout(Test)

Vaka (hasta) düzeyinde, `Bilgi.xlsx`'teki Bounding Box anotasyonlarından her üst sınıfın
kaç vakada en az 1 kez göründüğü. Train/val split dengesinin ve holdout temsiliyetinin
kontrolü için.

In [9]:
# ── Sınıf Dağılımı — Train / Val / Holdout(Test) ──────────────────────────
_bilgi_path = Path(os.environ.get('ABDOMEN_BILGI_XLSX', str(DATA_DIR / 'Bilgi.xlsx')))
if not _bilgi_path.exists():
    _candidates = list(DATA_DIR.rglob('Bilgi.xlsx'))
    if _candidates:
        _bilgi_path = _candidates[0]
    else:
        raise FileNotFoundError(f'Bilgi.xlsx bulunamadı: {_bilgi_path}')

_bilgi_sheets = pd.read_excel(_bilgi_path, sheet_name=None)
_train_raw = _bilgi_sheets['TRAIININGDATA'].copy()
_train_raw['Case Number'] = _train_raw['Case Number'].apply(lambda x: f'T_{x}')
_comp_raw  = _bilgi_sheets['COMPETITIONDATA'].copy()
_comp_raw['Case Number']  = _comp_raw['Case Number'].apply(lambda x: f'C_{x}')
_bilgi_df = pd.concat([_train_raw, _comp_raw], ignore_index=True)
_bb_df = _bilgi_df[_bilgi_df['Type'] == 'Bounding Box'].copy()

from src.config import super_id_from_raw
_bb_df['super_id'] = _bb_df['Class'].map(super_id_from_raw)
_bb_df = _bb_df.dropna(subset=['super_id'])
_bb_df['super_id'] = _bb_df['super_id'].astype(int)

_case_super = (
    _bb_df.groupby('Case Number')['super_id']
    .apply(lambda s: set(s.tolist()))
)

def _class_dist(cases, name):
    n = len(cases)
    print(f'--- {name} (n={n}) ---')
    for sid, cname in enumerate(SUPER_CLASSES):
        cnt = sum(1 for c in cases if sid in _case_super.get(c, set()))
        pct = 100 * cnt / n if n else 0
        print(f'  {cname:<30} {cnt:>5}  ({pct:5.1f}%)')
    print()

_train_cases   = pd.read_csv(SPLIT_DIR / f'fold{FOLD}_train.csv')['Case Number'].astype(str).tolist()
_val_cases     = pd.read_csv(SPLIT_DIR / f'fold{FOLD}_val.csv')['Case Number'].astype(str).tolist()
_holdout_csv   = SPLIT_DIR / 'holdout.csv'
_holdout_cases = pd.read_csv(_holdout_csv)['Case Number'].astype(str).tolist() if _holdout_csv.exists() else []

_class_dist(_train_cases, f'TRAIN (fold{FOLD})')
_class_dist(_val_cases,   f'VAL (fold{FOLD})')
if _holdout_cases:
    _class_dist(_holdout_cases, 'HOLDOUT (test)')

--- TRAIN (fold0) (n=742) ---
  acute_cholecystitis              151  ( 20.4%)
  kidney_ureter_stone              159  ( 21.4%)
  acute_pancreatitis               132  ( 17.8%)
  aortic_aneurysm_dissection       141  ( 19.0%)
  acute_appendicitis                59  (  8.0%)
  acute_diverticulitis              14  (  1.9%)

--- VAL (fold0) (n=186) ---
  acute_cholecystitis               39  ( 21.0%)
  kidney_ureter_stone               41  ( 22.0%)
  acute_pancreatitis                33  ( 17.7%)
  aortic_aneurysm_dissection        35  ( 18.8%)
  acute_appendicitis                15  (  8.1%)
  acute_diverticulitis               4  (  2.2%)

--- HOLDOUT (test) (n=164) ---
  acute_cholecystitis               33  ( 20.1%)
  kidney_ureter_stone               34  ( 20.7%)
  acute_pancreatitis                29  ( 17.7%)
  aortic_aneurysm_dissection        31  ( 18.9%)
  acute_appendicitis                13  (  7.9%)
  acute_diverticulitis               3  (  1.8%)



---
## 9. Doğrulama

In [10]:
SPLIT_DIR

PosixPath('/Users/ramazanpolat/Desktop/datasets/abdomen/outputs/splits')

In [11]:
print('=' * 52)
print(f'  Faz 3a — Veri Hazırlık Özeti')
print('=' * 52)

_manifest_ok = (SPLIT_DIR / 'manifest.csv').exists()
_splits_ok   = (SPLIT_DIR / 'splits.csv').exists()
print(f'  manifest.csv : {"✓" if _manifest_ok else "✗ EKSİK"}')
print(f'  splits.csv   : {"✓" if _splits_ok else "✗ EKSİK"}')
print()

all_ok = True
for fold in range(N_FOLDS):
    train_img = DET_DATA_DIR / f'fold{fold}' / 'images' / 'train'
    val_img   = DET_DATA_DIR / f'fold{fold}' / 'images' / 'val'
    n_tr = len(list(train_img.glob('*.png'))) if train_img.exists() else 0
    n_vl = len(list(val_img.glob('*.png')))   if val_img.exists()   else 0
    ok = n_tr > 0
    if not ok:
        all_ok = False
    print(f'  fold{fold}: {n_tr:>6,} train · {n_vl:>5,} val  {"✓" if ok else "✗ EKSİK"}')

print('=' * 52)
if all_ok:
    print('  Tüm foldlar hazır — Faz3 çalıştırılabilir ✓')
else:
    print('  Bazı foldlar eksik — cell_det_export\'u tekrar çalıştırın')

  Faz 3a — Veri Hazırlık Özeti
  manifest.csv : ✓
  splits.csv   : ✓

  fold0: 20,964 train · 5,470 val  ✓
  fold1:      0 train ·     0 val  ✗ EKSİK
  fold2:      0 train ·     0 val  ✗ EKSİK
  fold3:      0 train ·     0 val  ✗ EKSİK
  fold4:      0 train ·     0 val  ✗ EKSİK
  Bazı foldlar eksik — cell_det_export'u tekrar çalıştırın
